# Headroom compression playbook

Two compression points from the Module 1 RAG lab (`hw1.ipynb`), measured **before / after** on the same data:

1. **RAG context blob** — the 5 retrieved chunks concatenated into the prompt (`RAGBase.build_context`). One-shot.
2. **Agentic search-tool outputs** — what `search()` returns each turn, which the agent **re-sends every loop**.

Compression engine: Headroom `compress()` with the Kompress-base text model (`headroom[ml]`).

**Headroom behavior relevant here:**
- User and system messages are protected by default; enable compression with `compress_user_messages=True` and `compress_system_messages=True`.
- JSON tool payloads use `SmartCrusher` (structural only). These chunks are prose + code, so savings come from the Kompress text path applied to each rendered text block sent to the model.
- Code-heavy content triggers default protections (`router:protected:recent_code`), yielding ~0% compression. Disabling `protect_recent` and `protect_analysis_context` routes content through the mixed router, which shortens prose and trims low-value code (~15% on this corpus). That can drop exact code strings, so each scenario includes an end-to-end answer check.

**Setup** (TLS-inspecting proxy or offline reuse):
- `truststore.inject_into_ssl()` — trust the system CA store for Hugging Face downloads.
- `HF_HUB_DISABLE_XET=1` — use the pure-Python Hub downloader (avoids xet/Rust path issues with custom CAs).
- `HF_TOKEN` in `.env` — authenticated Hub access.
- `HF_HUB_OFFLINE=1` in `.env` — serve Kompress from cache (~0.6 GB under `~/.cache/huggingface/`) after the first download.

In [1]:
import os

# Hugging Face download setup (TLS proxy / offline cache — see intro)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import truststore

truststore.inject_into_ssl()

import transformers

transformers.logging.set_verbosity_error()

from dotenv import load_dotenv

load_dotenv()

from headroom import compress

# Tokenizer for Headroom token accounting (lab LLM is separate, via OpenRouter)
HR_MODEL = "gpt-4o"

## Load the same data as the lab

Same source, chunking, and index as `hw1.ipynb` (chunk `size=2000`, `step=1000`).

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [f.parse() for f in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

len(documents), len(chunks)

(72, 295)

## Compression helper

Both scenarios compress the same unit: the **text block** about to be sent to the model.
User/system compression is enabled; Kompress-base handles prose. `CompressResult` reports
tokens before and after.

### Compression parameters

Headroom measures and compresses **tokens**, not characters. Parameters passed through
`compress_block`:

| Parameter | Value in this notebook | Effect |
| --- | --- | --- |
| `target_ratio` | `0.5` | Hint/cap for the router, not a hard target. On this prose+code corpus the mixed router caps at ~0.52 regardless of lower values (see stress test below). |
| `min_tokens_to_compress` | `50` | Skip blocks below this token count. |
| `protect_recent` | `0` | When >0, keeps the last N code-like messages verbatim. Default Headroom behavior (~4) yields ~0% on code-heavy RAG context. |
| `protect_analysis_context` | `False` | When `True`, protects code detected as analysis context. |
| `compress_user_messages` | `True` | Required to compress user-role text (off by default in Headroom). |
| `compress_system_messages` | `True` | Same for system/developer messages. |

The Kompress model loads on first `compress()` call (~60s). A cold call may return
`router:noop` (0%); the warmup in the helper cell below stabilizes later measurements.

Higher compression removes exact strings (`while True`, `break`) from context. Headroom
reverts a block when compression would inflate tokens (`Optimization inflated tokens ...;
reverting`). End-to-end answer checks below validate quality, not token counts alone.

In [3]:
def compress_block(
    text,
    target_ratio=0.5,
    protect_recent=0,
    protect_analysis_context=False,
):
    """Compress a text block as a single user message.

    Enables user/system compression; disables recent-code and analysis-context
    protection so the mixed router can act on prose+code RAG content.
    """
    return compress(
        [{"role": "user", "content": text}],
        model=HR_MODEL,
        compress_user_messages=True,
        compress_system_messages=True,
        min_tokens_to_compress=50,
        target_ratio=target_ratio,
        protect_recent=protect_recent,
        protect_analysis_context=protect_analysis_context,
    )


def render_results(search_results):
    """Same shape as RAGBase.build_context: filename + content per hit."""
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


# Warm up Kompress model (first load ~60s; avoids router:noop on later cells)
_warm = compress_block("warm up the kompress model " * 30)
print("model ready — warmup:", _warm.tokens_before, "->", _warm.tokens_after, _warm.transforms_applied)

model ready — warmup: 188 -> 137 ['router:text:0.50']


## Scenario A — RAG context blob

One-shot RAG: retrieve 5 chunks, concatenate into the prompt, compress, measure token
delta.

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"

results = chunk_index.search(query, num_results=5)
context = render_results(results)

# Headroom defaults: recent-code + analysis-context protection on
res_default = compress_block(
    context,
    target_ratio=0.5,
    protect_recent=4,
    protect_analysis_context=True,
)
# Protections disabled: mixed router compresses prose+code
resA = compress_block(context, target_ratio=0.5)

print("RAG context blob")
print(f"  default protections: {res_default.tokens_before} -> {res_default.tokens_after} "
      f"({res_default.compression_ratio:.0%})  {res_default.transforms_applied}")
print(f"  protections off: {resA.tokens_before} -> {resA.tokens_after} "
      f"({resA.compression_ratio:.0%})  {resA.transforms_applied}")
print("\n--- compressed context (preview) ---")
print(resA.messages[0]["content"][:400])

RAG context blob
  default protections: 2224 -> 2224 (0%)  ['router:protected:recent_code']
  exposed protections off: 2224 -> 1894 (15%)  ['router:mixed:0.52']

--- compressed context (first 400 chars) ---
01-agentic-rag/lessons/14-agentic-loop.md loop. loop keeps calling response without function calls. also keep iteration counter see round-trips happened.
[32 items compressed to 16. Retrieve more: hash=8e320d1076ecb0a7fed97e5f]

```python
1 print(f"iteration False response openai_client.responses.create( model="gpt-5.4-mini", input=messages, tools=[search_tool], messages.extend(response.output) in


## Index placement — raw vs compressed-before-index

`minsearch` is lexical: compressing before indexing can remove exact tokens used as search
keys (`while True`, `break`, etc.).

Compare retrieval on the 5 chunks from the main query:

- index A: raw retrieved chunks
- index B: same chunks after Headroom compression
- search both with code-sensitive diagnostic queries

Isolates **compression before indexing** without re-compressing the full corpus.

In [5]:
candidate_chunks = [{**doc, "chunk_id": str(i)} for i, doc in enumerate(results)]
compressed_candidate_chunks = []

for doc in candidate_chunks:
    compressed = compress_block(doc["content"], target_ratio=0.5).messages[0]["content"]
    compressed_candidate_chunks.append({**doc, "content": compressed})

raw_candidate_index = Index(text_fields=["content"], keyword_fields=["filename", "chunk_id"])
raw_candidate_index.fit(candidate_chunks)

compressed_candidate_index = Index(text_fields=["content"], keyword_fields=["filename", "chunk_id"])
compressed_candidate_index.fit(compressed_candidate_chunks)

terms = ["while True", "break", "has_function_calls", "function_call", "messages.extend"]
print("Exact term preservation across the retrieved chunks:")
for term in terms:
    raw_hits = sum(term in doc["content"] for doc in candidate_chunks)
    comp_hits = sum(term in doc["content"] for doc in compressed_candidate_chunks)
    print(f"  {term:20} raw={raw_hits} compressed-before-index={comp_hits}")

diagnostic_queries = [
    "while True break",
    "has_function_calls False break",
    "function_call messages.extend",
    "iteration counter",
]

print("\nTop-3 retrieval comparison on raw vs compressed-before-index candidates:")
for q in diagnostic_queries:
    raw_top = raw_candidate_index.search(q, num_results=3)
    comp_top = compressed_candidate_index.search(q, num_results=3)
    print(f"\nquery: {q}")
    print("  raw index:       ", [(r["chunk_id"], r["filename"]) for r in raw_top])
    print("  compressed index:", [(r["chunk_id"], r["filename"]) for r in comp_top])

Optimization inflated tokens (325 -> 363); reverting to original messages


Optimization inflated tokens (473 -> 501); reverting to original messages


Exact term preservation across the retrieved chunks:
  while True           raw=3 compressed-before-index=1
  break                raw=3 compressed-before-index=0
  has_function_calls   raw=2 compressed-before-index=1
  function_call        raw=2 compressed-before-index=2
  messages.extend      raw=2 compressed-before-index=2

Top-3 retrieval comparison on raw vs compressed-before-index candidates:

query: while True break
  raw index:        [('0', '01-agentic-rag/lessons/14-agentic-loop.md'), ('1', '01-agentic-rag/lessons/14-agentic-loop.md'), ('4', '01-agentic-rag/lessons/15-frameworks.md')]
  compressed index: [('4', '01-agentic-rag/lessons/15-frameworks.md'), ('3', '01-agentic-rag/lessons/15-frameworks.md')]

query: has_function_calls False break
  raw index:        [('1', '01-agentic-rag/lessons/14-agentic-loop.md'), ('0', '01-agentic-rag/lessons/14-agentic-loop.md')]
  compressed index: [('0', '01-agentic-rag/lessons/14-agentic-loop.md'), ('1', '01-agentic-rag/lessons/14-agentic

## Scenario B — agentic search-tool outputs

In the agentic loop, each search appends tool output to the conversation; later turns
**re-send** all prior outputs. Token cost compounds across iterations.

Metrics below:

- **single-pass** — sum of compressed tool outputs, counted once each
- **loop-amplified** — output of call *k* billed `n_calls - k + 1` times

In [6]:
# The keyword searches an agent might fire for the same question.
agent_queries = [
    "agentic loop while True keep calling the model",
    "function_call tool execution append messages",
    "stop condition break no function calls",
    "difference between agentic loop and plain RAG",
]

per_call = []
for q in agent_queries:
    block = render_results(chunk_index.search(q, num_results=5))
    r = compress_block(block, target_ratio=0.5)
    per_call.append((r.tokens_before, r.tokens_after))

n = len(per_call)
single_before = sum(b for b, _ in per_call)
single_after = sum(a for _, a in per_call)

# Loop amplification: call k (1-indexed) is re-sent (n - k + 1) times.
amp_before = sum(b * (n - k) for k, (b, _) in enumerate(per_call))
amp_after = sum(a * (n - k) for k, (_, a) in enumerate(per_call))

print(f"agentic searches: {n}")
print("\nper-call (before -> after):")
for k, (b, a) in enumerate(per_call, 1):
    print(f"  call {k}: {b:>5} -> {a:<5} ({1 - a / b:.0%})")

print(f"\nsingle-pass total:   {single_before} -> {single_after}  ({1 - single_after / single_before:.0%})")
print(f"loop-amplified total: {amp_before} -> {amp_after}  ({1 - amp_after / amp_before:.0%})")
print(f"  (tokens the loop stops re-sending: {amp_before - amp_after})")

agentic searches: 4

per-call (before -> after):
  call 1:  2265 -> 1984  (12%)
  call 2:  2105 -> 1746  (17%)
  call 3:  2320 -> 1820  (22%)
  call 4:  2253 -> 2181  (3%)

single-pass total:   8943 -> 7731  (14%)
loop-amplified total: 22268 -> 18995  (15%)
  (tokens the loop stops re-sending: 3273)


## End-to-end eval

Same question, full vs compressed RAG context — compare answers and provider input
tokens.

In [7]:
from openai import OpenAI
from rag_helper import INSTRUCTIONS, PROMPT_TEMPLATE

client = OpenAI()
LLM_MODEL = "openai/gpt-5.4-mini"  # same model as the lab (via OpenRouter)


def answer_with_context(ctx_text):
    prompt = PROMPT_TEMPLATE.format(question=query, context=ctx_text)
    resp = client.responses.create(
        model=LLM_MODEL,
        input=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt},
        ],
        max_output_tokens=1024,
    )
    return resp.output_text, resp.usage


full_answer, full_usage = answer_with_context(context)
comp_answer, comp_usage = answer_with_context(resA.messages[0]["content"])

print(f"FULL context  -> input_tokens={full_usage.input_tokens}")
print(f"COMP context  -> input_tokens={comp_usage.input_tokens}")
print(f"saved at the provider: {full_usage.input_tokens - comp_usage.input_tokens} "
      f"({1 - comp_usage.input_tokens / full_usage.input_tokens:.0%})")
print("\n=== answer from FULL context ===\n", full_answer)
print("\n=== answer from COMPRESSED context ===\n", comp_answer)

FULL context  -> input_tokens=2294
COMP context  -> input_tokens=1964
saved at the provider: 330 (14%)

=== answer from FULL context ===
 It keeps calling the model inside a `while True` loop and checks each response for function calls.

- If the model returns a `function_call`, the code runs that tool, appends the tool result to `messages`, and loops again.
- If the model returns a normal `message` with no function calls, `has_function_calls` stays `False` and the loop breaks.

So the stopping rule is: **no function calls in the current turn means the agent is done**.

=== answer from COMPRESSED context ===
 The agentic loop keeps calling the model by:

1. Sending the current `messages` history to the model.
2. Checking the model output for tool/function calls.
3. If there is a function call, executing it and appending the tool result back into `messages`.
4. Repeating the loop.

It stops when the model returns an assistant message **with no function calls**. The simplest exit conditi

## `target_ratio` stress test

On this corpus, `target_ratio` is a hint the router can ignore: `0.5`, `0.2`, and `0.1`
all produce `router:mixed:0.52` (`2224 → 1894`).

The cell below compares `0.5` vs `0.2` for:
- token savings (identical on this content), and
- answer quality and key-fact preservation in context and LLM output.

For prose+code RAG context, protections (what content is eligible) matter more than
`target_ratio`.

In [8]:
key_facts = ["while True", "break", "function_call", "has_function_calls", "messages.extend"]


def fact_check(text):
    return {fact: (fact in text) for fact in key_facts}


resA_02 = compress_block(context, target_ratio=0.2)
agg_answer, agg_usage = answer_with_context(resA_02.messages[0]["content"])

print("RAG context blob compression by target_ratio")
print(f"  target_ratio=0.5: {resA.tokens_before} -> {resA.tokens_after} "
      f"({resA.compression_ratio:.0%})  {resA.transforms_applied}")
print(f"  target_ratio=0.2: {resA_02.tokens_before} -> {resA_02.tokens_after} "
      f"({resA_02.compression_ratio:.0%})  {resA_02.transforms_applied}")

print("\nProvider input tokens (end-to-end):")
print(f"  full:             {full_usage.input_tokens}")
print(f"  ratio=0.5:        {comp_usage.input_tokens} ({1 - comp_usage.input_tokens / full_usage.input_tokens:.0%} saved)")
print(f"  ratio=0.2:        {agg_usage.input_tokens} ({1 - agg_usage.input_tokens / full_usage.input_tokens:.0%} saved)")

print("\nKey-fact preservation in the compressed CONTEXT (literal substring):")
print(f"  ratio=0.5: {fact_check(resA.messages[0]['content'])}")
print(f"  ratio=0.2: {fact_check(resA_02.messages[0]['content'])}")

print("\nKey-fact coverage in the answer:")
print(f"  full:      {fact_check(full_answer)}")
print(f"  ratio=0.2: {fact_check(agg_answer)}")

print("\n--- compressed context @ target_ratio=0.2 (preview) ---")
print(resA_02.messages[0]["content"][:400])
print("\n=== answer from target_ratio=0.2 context ===\n", agg_answer)

RAG context blob compression by target_ratio
  target_ratio=0.5: 2224 -> 1894 (15%)  ['router:mixed:0.52']
  target_ratio=0.2: 2224 -> 1894 (15%)  ['router:mixed:0.52']

Provider input tokens (end-to-end):
  full:             2294
  ratio=0.5:        1964 (14% saved)
  ratio=0.2:        1964 (14% saved)

Key-fact preservation in the compressed CONTEXT (literal substring):
  ratio=0.5: {'while True': False, 'break': False, 'function_call': True, 'has_function_calls': True, 'messages.extend': True}
  ratio=0.2: {'while True': False, 'break': False, 'function_call': True, 'has_function_calls': True, 'messages.extend': True}

Key-fact coverage in the ANSWER (did the LLM still say it?):
  full:      {'while True': True, 'break': True, 'function_call': True, 'has_function_calls': True, 'messages.extend': False}
  ratio=0.2: {'while True': False, 'break': False, 'function_call': True, 'has_function_calls': True, 'messages.extend': False}

--- compressed context @0.2 (first 400 chars) ---
01-a

## Summary

- **Index raw chunks; compress after retrieval.** Pre-index compression drops lexical tokens (`while True`, `break`) and shifts retrieval for code-sensitive queries.
- **~14–15% savings** on RAG context and agentic tool outputs with protections disabled — consistent with Headroom benchmarks for short, dense, code-heavy content (not 60–95%).
- **Protections dominate `target_ratio`** on this corpus: `0.5`, `0.2`, and `0.1` all route to `router:mixed:0.52`. Code/user protection flags change eligibility; ratio does not.
- **End-to-end eval required** for code-detail questions: literal strings disappear from compressed context even when answers remain correct.
- **Optional layer:** the lab runs unchanged without compression; enable only when savings are measured and quality is verified.

**CCR:** This notebook uses library mode (`compress()` + direct `OpenAI()`). CCR markers (`Retrieve more: hash=...`) appear in compressed context but recovery is not wired — no `headroom_retrieve` tool on the model. For reversible compression, use Headroom proxy, `HeadroomClient`, or MCP retrieval.

In [9]:
rows = [
    ("A. RAG context blob (Kompress)", resA.tokens_before, resA.tokens_after),
    ("B. agentic outputs (single-pass)", single_before, single_after),
    ("B. agentic outputs (loop-amplified)", amp_before, amp_after),
    ("A. end-to-end provider input tokens", full_usage.input_tokens, comp_usage.input_tokens),
]

print(f"{'scenario':38} {'before':>8} {'after':>8} {'saved':>8}")
print("-" * 66)
for name, b, a in rows:
    print(f"{name:38} {b:>8} {a:>8} {1 - a / b:>7.0%}")

scenario                                 before    after    saved
------------------------------------------------------------------
A. RAG context blob (Kompress)             2224     1894     15%
B. agentic outputs (single-pass)           8943     7731     14%
B. agentic outputs (loop-amplified)       22268    18995     15%
A. end-to-end provider input tokens        2294     1964     14%
